# ADS-B I/Q physical-layer transmitter identification with K-fold validation

This is the same broad experiment as the earlier ADS-B I/Q notebook, but adapted to the new single-CSV data format. Each row is treated as one decoded ADS-B record with an associated raw I/Q capture in the `rawIQ` column. The decoded fields are only used for labels, grouping, filtering, and checking the data.

The working idea is to see whether a model can learn repeatable physical-layer differences between aircraft/transponder emissions directly from raw I/Q. I am avoiding decoded message contents as model inputs, because that would turn the task into reading identifiers rather than learning signal-level behaviour.

The basic flow is:

1. load the single CSV file,
2. parse the `rawIQ` strings into complex I/Q vectors,
3. use the aircraft ICAO value as the supervised class label,
4. crop or pad each I/Q vector to a fixed length,
5. normalise each record,
6. train a compact 1D CNN with stratified K-fold validation.

In [ ]:
# Optional setup
# I only need this if the environment is missing a package.
# %pip install numpy pandas scikit-learn matplotlib torch tqdm


In [ ]:
# Configuration
from pathlib import Path

CONFIG = {
    # New source format: one CSV with decoded ADS-B fields and a rawIQ column.
    "csv_path": Path("./adsb_records1.csv"),

    # The sample file has 976 complex samples per record. I am keeping this as the default
    # because it avoids unnecessary padding for the new data source.
    "window_len": 976,
    "normalise_per_record": True,
    "demean": True,

    # Label and filtering choices.
    # ICAO is not used as an input feature. It is only the training label.
    "label_column": "icao",
    "min_records_per_class": 5,
    "max_records_per_class": None,    # set to an int to balance/debug on large files
    "required_df": 17,                # set to None if I want to keep non-DF17 records too
    "min_snr": None,                  # can be set later if noisy rows dominate

    # K-fold settings.
    "n_splits": 5,
    "random_seed": 42,

    # Model settings.
    "num_filters_1": 32,
    "num_filters_2": 64,
    "num_filters_3": 96,
    "kernel_size": 7,
    "dropout_conv": 0.15,
    "dropout_fc": 0.35,
    "fc_units": 128,

    # Training settings.
    "batch_size": 64,
    "epochs": 15,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 4,

    # Outputs.
    "artifact_dir": Path("./adsb_iq_csv_outputs"),
}

CONFIG["artifact_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


## Data setup

The CSV is slightly awkward because the header and rows are written in a Python-list-like style, and both `bitStr` and `rawIQ` contain commas inside quoted fields. I am using Python's CSV parser with a single quote as the quote character so the embedded commas stay inside those fields.

The first attached sample only has a few records, so it is mainly useful for checking the parser and the input shape. The full file should have enough examples per aircraft for the cross-validation section.


In [ ]:
# Imports and reproducibility
import csv
import json
import math
import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["random_seed"])


In [ ]:
# CSV loading for the new ADS-B I/Q record format

NUMERIC_COLUMNS = [
    "num_msgs", "df", "timestamp", "altitude", "longitude", "heading",
    "snr", "latitude", "speed", "vertical_rate",
]


def _clean_header_field(value: str) -> str:
    return value.strip().strip("[]").strip().strip("'\"")


def load_adsb_record_csv(path: Path) -> pd.DataFrame:
    path = Path(path)

    with path.open(newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f, quotechar="'", skipinitialspace=True)
        raw_header = next(reader)
        header = [_clean_header_field(x) for x in raw_header]

        rows = []
        bad_rows = 0
        for row in reader:
            if len(row) != len(header):
                bad_rows += 1
                continue
            rows.append(row)

    df = pd.DataFrame(rows, columns=header)

    # Convert the usual numeric fields. The string values "nan" and "None" appear in the sample.
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].replace({"None": np.nan, "nan": np.nan, "": np.nan}),
                errors="coerce",
            )

    for col in ["icao", "callsign", "datetime", "bitStr", "rawIQ"]:
        if col in df.columns:
            df[col] = df[col].replace({"None": np.nan, "nan": np.nan, "": np.nan})

    if bad_rows:
        print(f"Skipped {bad_rows} malformed rows while reading {path.name}")

    return df


records = load_adsb_record_csv(CONFIG["csv_path"])

print("Loaded records:", records.shape)
display(records.head())
print(records.dtypes)


In [ ]:
# Quick data check
label_col = CONFIG["label_column"]

summary_cols = [c for c in ["icao", "df", "snr", "timestamp", "altitude", "latitude", "longitude", "speed", "vertical_rate"] if c in records.columns]
display(records[summary_cols].head())

if label_col in records.columns:
    class_counts = records[label_col].value_counts(dropna=False)
    display(class_counts.head(20).rename("records").to_frame())

if "rawIQ" in records.columns:
    raw_lengths = records["rawIQ"].dropna().map(lambda s: len(str(s).split(",")))
    display(raw_lengths.describe().rename("rawIQ sample count").to_frame())


In [ ]:
# I/Q parsing and record-level preparation

def parse_raw_iq(raw_iq) -> np.ndarray:
    if pd.isna(raw_iq):
        raise ValueError("Missing rawIQ")

    iq = np.fromstring(str(raw_iq), sep=",", dtype=np.complex64)
    if iq.size == 0:
        raise ValueError("rawIQ did not parse into complex samples")
    return iq


def crop_or_pad_iq(iq: np.ndarray, window_len: int) -> np.ndarray:
    iq = np.asarray(iq, dtype=np.complex64).reshape(-1)

    if len(iq) == window_len:
        return iq

    if len(iq) > window_len:
        # Centre crop so timing drift at either edge is less likely to dominate.
        start = (len(iq) - window_len) // 2
        return iq[start:start + window_len]

    # Zero padding keeps the model input fixed when a small number of rows are short.
    out = np.zeros(window_len, dtype=np.complex64)
    out[:len(iq)] = iq
    return out


def normalise_iq(iq: np.ndarray, demean: bool = True) -> np.ndarray:
    iq = np.asarray(iq, dtype=np.complex64)

    if demean:
        iq = iq - np.mean(iq)

    scale = np.sqrt(np.mean(np.abs(iq) ** 2)) + 1e-12
    return (iq / scale).astype(np.complex64)


def iq_to_channels(iq: np.ndarray) -> np.ndarray:
    # PyTorch Conv1d expects channels first: [I/Q channel, sample index].
    return np.stack([iq.real, iq.imag], axis=0).astype(np.float32)


def build_record_dataset(records: pd.DataFrame, config=CONFIG):
    df = records.copy()

    if config["required_df"] is not None and "df" in df.columns:
        df = df[df["df"] == config["required_df"]].copy()

    if config["min_snr"] is not None and "snr" in df.columns:
        df = df[df["snr"] >= config["min_snr"]].copy()

    df = df.dropna(subset=[config["label_column"], "rawIQ"]).copy()
    df[config["label_column"]] = df[config["label_column"]].astype(str)

    counts = df[config["label_column"]].value_counts()
    keep_labels = counts[counts >= config["min_records_per_class"]].index
    df = df[df[config["label_column"]].isin(keep_labels)].copy()

    if config["max_records_per_class"] is not None:
        df = (
            df.groupby(config["label_column"], group_keys=False)
              .sample(n=config["max_records_per_class"], random_state=config["random_seed"], replace=False)
        )

    rows = []
    parse_errors = 0

    for row_idx, row in df.iterrows():
        try:
            iq = parse_raw_iq(row["rawIQ"])
            iq = crop_or_pad_iq(iq, config["window_len"])

            if config["normalise_per_record"]:
                iq = normalise_iq(iq, demean=config["demean"])

            rows.append({
                "x": iq_to_channels(iq),
                "y": row[config["label_column"]],
                "record_index": int(row_idx),
                "timestamp": row.get("timestamp", np.nan),
                "snr": row.get("snr", np.nan),
                "icao": row.get("icao", None),
            })
        except Exception:
            parse_errors += 1

    if parse_errors:
        print(f"Skipped {parse_errors} rows that could not be converted into I/Q vectors")

    if not rows:
        raise ValueError(
            "No usable I/Q records after filtering. "
            "For the small sample file, lower CONFIG['min_records_per_class'] just to test the parser."
        )

    return rows, df


rows, filtered_records = build_record_dataset(records, CONFIG)

print("Usable records:", len(rows))
print("Filtered table shape:", filtered_records.shape)
print("Classes kept:", filtered_records[CONFIG["label_column"]].nunique())
display(filtered_records[CONFIG["label_column"]].value_counts().rename("records").to_frame().head(20))


In [ ]:
# Encode labels and inspect the arrays
X = np.stack([r["x"] for r in rows])
y_text = np.array([r["y"] for r in rows])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = list(label_encoder.classes_)

print("X shape:", X.shape)          # records, I/Q channels, samples
print("y shape:", y.shape)
print("Number of classes:", len(class_names))

class_table = pd.DataFrame({
    "class": class_names,
    "encoded": np.arange(len(class_names)),
    "records": [int(np.sum(y == i)) for i in range(len(class_names))],
})
display(class_table.sort_values("records", ascending=False).head(30))


In [ ]:
# Quick sanity plots
# I am plotting only a few examples so the notebook stays readable.

def plot_examples(X, y, class_names, max_classes=4):
    chosen = []
    for class_idx in range(min(len(class_names), max_classes)):
        indices = np.where(y == class_idx)[0]
        if len(indices):
            chosen.append(indices[0])

    for idx in chosen:
        plt.figure(figsize=(12, 3))
        plt.plot(X[idx, 0], label="I")
        plt.plot(X[idx, 1], label="Q")
        plt.title(f"Example ADS-B I/Q record - class {class_names[y[idx]]}")
        plt.xlabel("Sample")
        plt.ylabel("Normalised amplitude")
        plt.legend()
        plt.tight_layout()
        plt.show()


plot_examples(X, y, class_names)


## Model

For this baseline I am using a compact 1D CNN on raw I/Q samples. This keeps the experiment close to the earlier notebook while removing the old folder/burst-detection loader. The model sees only the normalised I/Q channels, not `icao`, `bitStr`, `callsign`, position, altitude, or other decoded ADS-B fields.

The immediate goal is not to claim operational reliability. It is to get a repeatable offline baseline that can later be tested against stronger splits, extra receiver positions, and smaller models for real-time RFSoC work \cite{soltani2019realtimeembeddeddeeplearning}.


In [ ]:
# PyTorch dataset and model

class IQRecordDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class ADSBIQCNN(nn.Module):
    def __init__(self, num_classes: int, window_len: int):
        super().__init__()
        k = CONFIG["kernel_size"]
        pad = k // 2

        self.features = nn.Sequential(
            nn.Conv1d(2, CONFIG["num_filters_1"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_1"]),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CONFIG["dropout_conv"]),

            nn.Conv1d(CONFIG["num_filters_1"], CONFIG["num_filters_2"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_2"]),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CONFIG["dropout_conv"]),

            nn.Conv1d(CONFIG["num_filters_2"], CONFIG["num_filters_3"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_3"]),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(CONFIG["num_filters_3"], CONFIG["fc_units"]),
            nn.ReLU(),
            nn.Dropout(CONFIG["dropout_fc"]),
            nn.Linear(CONFIG["fc_units"], num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model_preview = ADSBIQCNN(num_classes=len(class_names), window_len=CONFIG["window_len"])
print(model_preview)


In [ ]:
# Training and evaluation helpers

def make_loaders(train_idx, val_idx):
    train_ds = IQRecordDataset(X[train_idx], y[train_idx])
    val_ds = IQRecordDataset(X[val_idx], y[val_idx])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    return train_loader, val_loader


def train_one_epoch(model, loader, criterion, optimiser):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimiser.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimiser.step()

        total_loss += loss.item() * len(yb)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds = []
    targets = []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * len(yb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        targets.append(yb.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro", zero_division=0),
        "preds": preds,
        "targets": targets,
    }


In [ ]:
# Stratified K-fold cross-validation
# This is record-level validation. A stronger later test should split by collection
# session, receiver location, day, or aircraft pass if that metadata is available.

class_counts = Counter(y)
min_class_count = min(class_counts.values())
n_splits = min(CONFIG["n_splits"], min_class_count)

sufficient_validation_data = len(class_names) >= 2 and n_splits >= 2

fold_results = []
all_fold_preds = []
all_fold_targets = []

if not sufficient_validation_data:
    print(
        "Not enough class coverage for K-fold validation in the currently loaded file. "
        "The attached small CSV is still useful for checking the parser and I/Q shape. "
        "Use the full CSV, or lower filtering only for debugging, before training."
    )
else:
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=CONFIG["random_seed"],
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        print(f"\n===== Fold {fold}/{n_splits} =====")
        set_seed(CONFIG["random_seed"] + fold)

        train_loader, val_loader = make_loaders(train_idx, val_idx)
        model = ADSBIQCNN(num_classes=len(class_names), window_len=CONFIG["window_len"]).to(DEVICE)

        criterion = nn.CrossEntropyLoss()
        optimiser = torch.optim.AdamW(
            model.parameters(),
            lr=CONFIG["learning_rate"],
            weight_decay=CONFIG["weight_decay"],
        )

        best_state = None
        best_val_loss = float("inf")
        epochs_without_improvement = 0

        for epoch in range(1, CONFIG["epochs"] + 1):
            train_loss = train_one_epoch(model, train_loader, criterion, optimiser)
            val_metrics = evaluate(model, val_loader, criterion)

            print(
                f"Epoch {epoch:02d} | "
                f"train loss {train_loss:.4f} | "
                f"val loss {val_metrics['loss']:.4f} | "
                f"val acc {val_metrics['accuracy']:.4f} | "
                f"val macro-F1 {val_metrics['macro_f1']:.4f}"
            )

            if val_metrics["loss"] < best_val_loss:
                best_val_loss = val_metrics["loss"]
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= CONFIG["patience"]:
                print("Stopping early for this fold.")
                break

        if best_state is not None:
            model.load_state_dict(best_state)

        final_metrics = evaluate(model, val_loader, criterion)

        fold_results.append({
            "fold": fold,
            "val_loss": final_metrics["loss"],
            "val_accuracy": final_metrics["accuracy"],
            "val_macro_f1": final_metrics["macro_f1"],
            "n_train": len(train_idx),
            "n_val": len(val_idx),
        })

        all_fold_preds.append(final_metrics["preds"])
        all_fold_targets.append(final_metrics["targets"])

results_df = pd.DataFrame(fold_results)
display(results_df)


In [ ]:
# Summarise results

if results_df.empty:
    print("No training results to summarise yet. Load the full CSV with at least two usable classes and rerun from the data cells.")
else:
    display(results_df.describe())

    all_preds = np.concatenate(all_fold_preds)
    all_targets = np.concatenate(all_fold_targets)

    print("\nOverall classification report:")
    print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, aspect="auto")
    plt.title("ADS-B physical-layer transmitter ID confusion matrix")
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.colorbar(label="Count")
    plt.tight_layout()
    plt.show()

    results_df.to_csv(CONFIG["artifact_dir"] / "kfold_results.csv", index=False)

    with open(CONFIG["artifact_dir"] / "label_mapping.json", "w") as f:
        json.dump({int(i): name for i, name in enumerate(class_names)}, f, indent=2)

    print("Saved artefacts to:", CONFIG["artifact_dir"].resolve())


## Notes for next steps

This is still a baseline experiment, not the final real-time RFSoC version.

Things I would check next:

1. use a stricter split if the full dataset has collection-session metadata, because random record-level splits can overstate performance;
2. keep decoded identifiers and aircraft-state fields out of the model inputs, using them only for labels and sanity checks;
3. compare results with and without very low-SNR records so the model is not just learning capture quality;
4. test a smaller window length to see how little raw I/Q is needed before accuracy collapses;
5. add an unknown-class rejection step, because a practical system should not force every unseen aircraft into a known class;
6. once the offline result is stable, shrink the model with shorter windows, pruning, or quantisation-aware training for the ZCU111/RFSoC direction.
